# 1 Module importieren

In [1]:
import pandas as pd

import m2cgen as m2c

from scipy.stats import uniform

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

from xgboost import XGBClassifier


# 2 Daten Importieren

## 2.1 AGMP

In [2]:
agmp_dataPath = 'datasets/agmp/'
##### Data Import and Splitting #####
agmp_data = pd.read_csv(agmp_dataPath + 'gyro_mobile.csv')
agmp_data = agmp_data.drop(columns='timestamp')
agmp_xdata = agmp_data.iloc[:,:6]
agmp_ydata = agmp_data.iloc[:,6:]

agmp_xtrain, agmp_xvaltest, agmp_ytrain, agmp_yvaltest = train_test_split( 
    agmp_xdata,
    agmp_ydata,
    random_state=0,
    train_size=0.66,
    stratify=agmp_ydata
)

agmp_xval, agmp_xtest, agmp_yval, agmp_ytest = train_test_split( 
    agmp_xvaltest,
    agmp_yvaltest,
    random_state=0,
    train_size=0.5,
    stratify=agmp_yvaltest
)

# Zuvor gesicherte Datensätze nach RFECV werden importiert

agmp_xtrain = pd.read_csv("datasets/agmp/feature_selection/agmp_xtrain_rfecv.csv", sep=",")
agmp_xval = pd.read_csv("datasets/agmp/feature_selection/agmp_xval_rfecv.csv", sep=",")
agmp_xtest = pd.read_csv("datasets/agmp/feature_selection/agmp_xtest_rfecv.csv", sep=",")

classratio = len(agmp_data[agmp_data['Activity']==0]) / len(agmp_data[agmp_data['Activity']==1])

agmp_ev_val = [(agmp_xval,agmp_yval)]
agmp_ev_all = [(agmp_xtrain,agmp_ytrain),(agmp_xval,agmp_yval),(agmp_xtest,agmp_ytest)]

## 2.2 HARUS

In [3]:
harus_dataPath = 'datasets/harus/'

harus_xdata = pd.read_csv(harus_dataPath + "xdata.csv", sep=";")                     
harus_ydata = pd.read_csv(harus_dataPath + "ydata.csv", sep=";")

harus_xtrain, harus_xvaltest, harus_ytrain, harus_yvaltest = train_test_split( 
    harus_xdata,
    harus_ydata,
    random_state=0,
    train_size=0.66,
    stratify=harus_ydata
)

harus_xval, harus_xtest, harus_yval, harus_ytest = train_test_split( 
    harus_xvaltest,
    harus_yvaltest,
    random_state=0,
    train_size=0.5,
    stratify=harus_yvaltest
)

# Zuvor gesicherte Datensätze nach RFECV werden importiert

harus_xtrain = pd.read_csv("datasets/harus/feature_selection/harus_xtrain_rfecv.csv", sep=",")
harus_xval = pd.read_csv("datasets/harus/feature_selection/harus_xval_rfecv.csv", sep=",")
harus_xtest = pd.read_csv("datasets/harus/feature_selection/harus_xtest_rfecv.csv", sep=",")

harus_ev_val = [(harus_xval,harus_yval)]
harus_ev_all = [(harus_xtrain,harus_ytrain),(harus_xval,harus_yval),(harus_xtest,harus_ytest)]

## 2.3 SeMu

In [4]:
semu_dataPath = 'datasets/semu/'                         # Set location of dataset

semu_data = pd.read_csv(semu_dataPath + 'secondary_mushroom.csv', sep=";")    # Dataset is imbalanced with only ~1.7% of all labels being 0's
semu_xdata = semu_data.iloc[:,:20]
semu_ydata = semu_data.iloc[:,20:]    # Classratio 9:11

def encodeLabels():
    encoders = {}

    # Create a list with the names of all columns, that contain categorical data
    xdata_cats = list(semu_xdata.columns)
    [xdata_cats.remove(i) for i in ['cap-diameter','stem-height','stem-width']]

    # Encode all categorical feature-labels and save encoders in a dictionary
    for cat in xdata_cats:
        le = LabelEncoder()
        semu_xdata[cat] = le.fit_transform(semu_xdata[cat])
        encoders.update({cat:le})

    le = LabelEncoder()
    semu_ydata['class'] = le.fit_transform(semu_ydata['class'])
    encoders.update({'class':le})

    return encoders

def encodeOHE():
    xdata_categorical = semu_xdata.copy()
    numeric_features = ['cap-diameter','stem-height','stem-width']

    for i in numeric_features:
        xdata_categorical = xdata_categorical.drop(i,axis=1) 

    xdata_numeric = pd.DataFrame()
    for feat in numeric_features:
        xdata_numeric[feat] = semu_xdata[feat]

    ohe = OneHotEncoder()
    ohedata = ohe.fit_transform(xdata_categorical)

    xdata_categorical_ohe = pd.DataFrame(ohedata.toarray(), columns=ohe.get_feature_names_out(), dtype=int)

    xdata_encoded = pd.concat([xdata_numeric, xdata_categorical_ohe],axis=1)

    yEncoder = LabelEncoder()
    semu_ydata['class'] = yEncoder.fit_transform(semu_ydata['class'])

    return xdata_encoded, yEncoder

semu_xdata, yEncoder = encodeOHE() # Weil keine Rangfolge zwischen den einzelnen Kategorien herrscht, wird OHE angewendet

semu_xtrain, semu_xvaltest, semu_ytrain, semu_yvaltest = train_test_split( 
    semu_xdata,
    semu_ydata,
    random_state=0,
    train_size=0.66,
    stratify=semu_ydata
)

semu_xval, semu_xtest, semu_yval, semu_ytest = train_test_split( 
    semu_xvaltest,
    semu_yvaltest,
    random_state=0,
    train_size=0.5,
    stratify=semu_yvaltest
)

# Zuvor gesicherte Datensätze nach RFECV werden importiert

semu_xtrain = pd.read_csv("datasets/semu/feature_selection/semu_xtrain_rfecv.csv", sep=",")
semu_xval = pd.read_csv("datasets/semu/feature_selection/semu_xval_rfecv.csv", sep=",")
semu_xtest = pd.read_csv("datasets/semu/feature_selection/semu_xtest_rfecv.csv", sep=",")

semu_ev_val = [(semu_xval,semu_yval)]
semu_ev_all = [(semu_xtrain,semu_ytrain),(semu_xval,semu_yval),(semu_xtest,semu_ytest)]

# 3 Suchraum definieren

In [5]:
searchspace = {
    'max_depth': [1,3,5,7,9],
    'n_estimators': [100,200,300,400,500,600,700,800,900,1000],
    'min_child_weight': [1,3,5,7],
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'learning_rate': uniform(0.01, 0.29),
    'gamma': uniform(0.1, 0.8)
}

# 4 Hyperparameter optimieren mit Random Search

## 4.1 AGMP

In [6]:
agmp_xgb = XGBClassifier(
    objective='binary:logistic',
    tree_method='exact',
    scale_pos_weight=classratio,
    n_jobs=-1,
    early_stopping_rounds=10,
)

agmp_random_search = RandomizedSearchCV(
    estimator=agmp_xgb, 
    param_distributions=searchspace, 
    scoring='balanced_accuracy',
    n_iter=250, 
    cv=5,
    n_jobs=-1, 
    verbose=1, 
    random_state=0
)
agmp_random_search.fit(
    agmp_xtrain, agmp_ytrain,
    eval_set=agmp_ev_val,
    verbose=True
)

# Print best parameters
print(f"Best parameters: {agmp_random_search.best_params_}")
print(f"Best score: {agmp_random_search.best_score_}")

Fitting 5 folds for each of 250 candidates, totalling 1250 fits
[0]	validation_0-logloss:0.63134[0]	validation_0-logloss:0.68421
[0]	validation_0-logloss:0.63620
[0]	validation_0-logloss:0.63708

[0]	validation_0-logloss:0.64407
[1]	validation_0-logloss:0.58785
[1]	validation_0-logloss:0.59170
[1]	validation_0-logloss:0.67590
[1]	validation_0-logloss:0.58654
[1]	validation_0-logloss:0.58719
[2]	validation_0-logloss:0.55360
[2]	validation_0-logloss:0.55813
[2]	validation_0-logloss:0.66673
[2]	validation_0-logloss:0.55179
[2]	validation_0-logloss:0.55592
[3]	validation_0-logloss:0.53061[3]	validation_0-logloss:0.53562

[3]	validation_0-logloss:0.52758
[3]	validation_0-logloss:0.65802
[3]	validation_0-logloss:0.52145
[0]	validation_0-logloss:0.68324
[4]	validation_0-logloss:0.51606
[4]	validation_0-logloss:0.51243
[4]	validation_0-logloss:0.50865
[4]	validation_0-logloss:0.64965
[4]	validation_0-logloss:0.50106
[1]	validation_0-logloss:0.67502
[0]	validation_0-logloss:0.60549
[5]	validati

## 4.2 HARUS


In [7]:
harus_xgb = XGBClassifier(
    objective='multi:softmax',
    tree_method='exact',
    n_jobs=-1,
    early_stopping_rounds=10
)

harus_random_search = RandomizedSearchCV(
    estimator=harus_xgb, 
    param_distributions=searchspace, 
    scoring='accuracy',
    n_iter=250, 
    cv=5,
    n_jobs=-1, 
    verbose=1, 
    random_state=0
)
harus_random_search.fit(harus_xtrain, harus_ytrain,
                  eval_set = harus_ev_val,
                  verbose=False)

# Print best parameters
print(f"Best parameters: {harus_random_search.best_params_}")
print(f"Best score: {harus_random_search.best_score_}")

Fitting 5 folds for each of 250 candidates, totalling 1250 fits


KeyboardInterrupt: 

## 4.3 SeMu

In [ ]:
semu_xgb = XGBClassifier(
    objective='binary:logistic',
    tree_method='exact',
    n_jobs=-1,
    early_stopping_rounds=10,
)

semu_random_search = RandomizedSearchCV(
    estimator=semu_xgb, 
    param_distributions=searchspace, 
    scoring='accuracy',
    n_iter=250, 
    cv=5,
    n_jobs=-1, 
    verbose=1, 
    random_state=0
)
semu_random_search.fit(
    semu_xtrain, semu_ytrain,
    eval_set=semu_ev_val,
    verbose=True
)

# Print best parameters
print(f"Best parameters: {semu_random_search.best_params_}")
print(f"Best score: {semu_random_search.best_score_}")


# 5 Modelle mit optimierten Hyperparametern erzeugen

## 5.1 AGMP


In [8]:
pre_agmp = XGBClassifier(
    objective='binary:logistic',
    tree_method='exact',
    n_jobs=-1,
    early_stopping_rounds=10,
    scale_pos_weight=classratio,
    n_estimators=agmp_random_search.best_params_['n_estimators'],
    max_depth=agmp_random_search.best_params_['max_depth'],
    min_child_weight=agmp_random_search.best_params_['min_child_weight'],
    subsample=agmp_random_search.best_params_['subsample'],
    colsample_bytree=agmp_random_search.best_params_['colsample_bytree'],
    learning_rate=agmp_random_search.best_params_['learning_rate'],
    gamma=agmp_random_search.best_params_['gamma'],
)

pre_agmp.fit(
    agmp_xtrain, 
    agmp_ytrain,
    eval_set=agmp_ev_val,
    verbose=False
)

agmp_best_iteration = pre_agmp.best_iteration
print(f'n_estimators nach best_iteration: {agmp_best_iteration}')

agmp = XGBClassifier(
    objective='binary:logistic',
    tree_method='exact',
    scale_pos_weight=classratio,
    n_jobs=-1,
    n_estimators=agmp_best_iteration,
    max_depth=agmp_random_search.best_params_['max_depth'],
    min_child_weight=agmp_random_search.best_params_['min_child_weight'],
    subsample=agmp_random_search.best_params_['subsample'],
    colsample_bytree=agmp_random_search.best_params_['colsample_bytree'],
    learning_rate=agmp_random_search.best_params_['learning_rate'],
    gamma=agmp_random_search.best_params_['gamma'],
)

agmp.fit(
    agmp_xtrain, agmp_ytrain
)

n_estimators nach best_iteration: 48


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=np.float64(0.761287213218212), device=None,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric=None, feature_types=None,
              gamma=np.float64(0.6064496054554777), grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=np.float64(0.2682893147260598), max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=7, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=48, n_jobs=-1,
              num_parallel_tree=None, random_state=None, ...)

## 5.2 HARUS


In [9]:
pre_harus = XGBClassifier(
    objective = "multi:softmax",
    tree_method = 'exact',
    early_stopping_rounds = 10,
    n_jobs = -1,
    n_estimators = harus_random_search.best_params_['n_estimators'],
    max_depth = harus_random_search.best_params_['max_depth'],
    min_child_weight = harus_random_search.best_params_['min_child_weight'],
    subsample = harus_random_search.best_params_['subsample'],
    colsample_bytree = harus_random_search.best_params_['colsample_bytree'],
    learning_rate = harus_random_search.best_params_['learning_rate'],
    gamma = harus_random_search.best_params_['gamma'],
)

pre_harus.fit(
    harus_xtrain, 
    harus_ytrain,
    eval_set=harus_ev_val,
    verbose=False
)

harus_best_iteration = pre_harus.best_iteration
print(f'n_estimators nach best_iteration: {harus_best_iteration}')

harus = XGBClassifier(
    objective = "multi:softmax",
    tree_method = 'exact',
    n_jobs = -1,
    n_estimators = harus_best_iteration,
    max_depth = harus_random_search.best_params_['max_depth'],
    min_child_weight = harus_random_search.best_params_['min_child_weight'],
    subsample = harus_random_search.best_params_['subsample'],
    colsample_bytree = harus_random_search.best_params_['colsample_bytree'],
    learning_rate = harus_random_search.best_params_['learning_rate'],
    gamma = harus_random_search.best_params_['gamma'],
)

harus.fit(
    harus_xtrain, harus_ytrain
)

AttributeError: 'RandomizedSearchCV' object has no attribute 'best_params_'

## 5.3 SeMu

In [ ]:
pre_semu = XGBClassifier(
    objective='binary:logistic',
    tree_method='exact',
    n_jobs=-1,
    early_stopping_rounds=10,
    n_estimators = semu_random_search.best_params_['n_estimators'],
    max_depth = semu_random_search.best_params_['max_depth'],
    min_child_weight = semu_random_search.best_params_['min_child_weight'],
    subsample = semu_random_search.best_params_['subsample'],
    colsample_bytree = semu_random_search.best_params_['colsample_bytree'],
    learning_rate = semu_random_search.best_params_['learning_rate'],
    gamma = semu_random_search.best_params_['gamma'],
)

pre_semu.fit(
    semu_xtrain, 
    semu_ytrain,
    eval_set=semu_ev_val,
    verbose=False
)

semu_best_iteration = pre_semu.best_iteration
print(f'n_estimators nach best_iteration: {semu_best_iteration}')

semu = XGBClassifier(
    objective='binary:logistic',
    tree_method='exact',
    n_jobs=-1,
    n_estimators = semu_best_iteration,
    max_depth = semu_random_search.best_params_['max_depth'],
    min_child_weight = semu_random_search.best_params_['min_child_weight'],
    subsample = semu_random_search.best_params_['subsample'],
    colsample_bytree = semu_random_search.best_params_['colsample_bytree'],
    learning_rate = semu_random_search.best_params_['learning_rate'],
    gamma = semu_random_search.best_params_['gamma'],
)
semu.fit(
    semu_xtrain, semu_ytrain
)

n_estimators nach best_iteration: 121


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.919231607902399, device=None,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric=None, feature_types=None, gamma=0.540795176148389,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.2943352730637827,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=9, max_leaves=None,
              min_child_weight=1, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=121, n_jobs=-1,
              num_parallel_tree=None, random_state=None, ...)

# 6 Ermittelte Hyperparameter sichern

In [10]:
hyperparam_quicksave = "Gefundene Hyperparameter:\n"
hyperparam_quicksave += f"##### Agmp #####\n {agmp_random_search.best_params_}\n"
hyperparam_quicksave += f"nach best_iteration: {agmp_best_iteration}\n"
hyperparam_quicksave += f"##### Harus #####\n {harus_random_search.best_params_}\n"
hyperparam_quicksave += f"nach best_iteration: {harus_best_iteration}\n"
hyperparam_quicksave += f"##### Semu #####\n {semu_random_search.best_params_}\n"
hyperparam_quicksave += f"nach best_iteration: {semu_best_iteration}\n"
with open("hyperparameter_quicksave.txt", "w") as file:
    file.write(hyperparam_quicksave)

AttributeError: 'RandomizedSearchCV' object has no attribute 'best_params_'

# 7 Modelle mit optimierten Parametern erzeugen

In [16]:
agmp_opt = XGBClassifier(
    objective='binary:logistic',
    tree_method='exact',
    scale_pos_weight=classratio,
    n_jobs=-1,
    max_depth=5,
    n_estimators=48,
    learning_rate=0.2682893147260598,
    min_child_weight=7,
    subsample=0.7826079565050621,
    colsample_bytree=0.761287213218212,
    gamma=0.6064496054554777,
)

harus_opt = XGBClassifier(
    objective='multi:softmax',
    tree_method='exact',
    n_jobs=-1,
    max_depth=3,
    n_estimators=249,
    learning_rate=0.13481670745733776,
    min_child_weight=1,
    subsample=0.6575317462724224,
    colsample_bytree=0.8582280977824015,
    gamma=0.12828994860439275,
)

semu_opt = XGBClassifier(
    objective='binary:logistic',
    tree_method='exact',
    n_jobs=-1,
    max_depth=5,
    n_estimators=499,
    learning_rate=0.15027487381296678,
    min_child_weight=3,
    subsample=0.7599248560037323,
    colsample_bytree=0.63998340623714,
    gamma=0.13172036479303575,
)

agmp_opt.fit(
    agmp_xtrain, agmp_ytrain
)
harus_opt.fit(
    harus_xtrain, harus_ytrain
)
semu_opt.fit(
    semu_xtrain, semu_ytrain
)

agmp_opt.save_model("model-exports/aktuelle-exports/combined/agmp_comb.json")
harus_opt.save_model("model-exports/aktuelle-exports/combined/harus_comb.json")
semu_opt.save_model("model-exports/aktuelle-exports/combined/semu_comb.json")

KeyboardInterrupt: 

# 8 Scores quantisieren

In [12]:
import json

def getScaleZP(modeldata, amountBits):
    min = 0.0
    max = 0.0
    
    for tree in modeldata['learner']['gradient_booster']['model']['trees']:
        nodescount = int(tree['tree_param']['num_nodes'])
        
        for i in range(nodescount):
            if int(tree['left_children'][i]) == -1:
                if float(tree['split_conditions'][i]) < min:
                    min = float(tree['split_conditions'][i])
                if float(tree['split_conditions'][i]) > max:
                    max = float(tree['split_conditions'][i])

    print(f"min: {min}, max: {max}")

    print(f'Amount of Bits: {amountBits} -> Skala: {(2**amountBits)-1} ')
    scale = ((2**amountBits)-1) / (max - min)
    zp = (-(round(scale*min))) - 128

    return scale, zp


def quantizeScores(modeldata, scale, zp):
    for tree in modeldata['learner']['gradient_booster']['model']['trees']:
        nodescount = int(tree['tree_param']['num_nodes'])
        
        for i in range(nodescount):
            if int(tree['left_children'][i]) == -1:
                score = float(tree['split_conditions'][i])
                tree['split_conditions'][i] = float(round(scale * score + zp))
                
    return modeldata

In [13]:
with open('model-exports/aktuelle-exports/combined/agmp_comb.json') as f:
    agmp_json = json.load(f)

with open('model-exports/aktuelle-exports/combined/harus_comb.json') as f:
    harus_json = json.load(f)

with open('model-exports/aktuelle-exports/combined/semu_comb.json') as f:
    semu_json = json.load(f)

agmp_scale, agmp_zp = getScaleZP(agmp_json, 8)
harus_scale, harus_zp = getScaleZP(harus_json, 8)
semu_scale, semu_zp = getScaleZP(semu_json, 8)

print(f"agmp_scale: {agmp_scale}, agmp_zp: {agmp_zp}")
print(f"harus_scale: {harus_scale}, harus_zp: {harus_zp}")
print(f"semu_scale: {semu_scale}, semu_zp: {semu_zp}")


agmp_json = quantizeScores(agmp_json, agmp_scale, agmp_zp)
harus_json = quantizeScores(harus_json, harus_scale, harus_zp)
semu_json = quantizeScores(semu_json, semu_scale, semu_zp)

min: -0.2923554, max: 0.55096006
Amount of Bits: 8 -> Skala: 255 
min: -0.1465948, max: 0.4027076
Amount of Bits: 8 -> Skala: 255 
min: -0.445063, max: 0.40438992
Amount of Bits: 8 -> Skala: 255 
agmp_scale: 302.37795000224475, agmp_zp: -40
harus_scale: 464.22516996102695, harus_zp: -60
semu_scale: 300.19321141423586, semu_zp: 6


In [14]:
with open('model-exports/aktuelle-exports/combined/agmp_comb.json', 'w') as f:
    json.dump(agmp_json, f, separators=(',', ':'))
with open('model-exports/aktuelle-exports/combined/harus_comb.json', 'w') as f:
    json.dump(harus_json, f, separators=(',', ':'))
with open('model-exports/aktuelle-exports/combined/semu_comb.json', 'w') as f:
    json.dump(semu_json, f, separators=(',', ':'))

# 9 Portieren

In [15]:
def portToC(model, filename):
    path = 'model-exports/plainC/'

    with open(path + filename,'w') as f:
        code = m2c.export_to_c(model)
        f.write(code)

    print(f'Model exported to: "{path + filename}"')

agmp_opt.load_model("model-exports/aktuelle-exports/combined/agmp_comb.json")
agmp_opt.set_params(base_score = float(agmp_opt.base_score))
print(float(agmp_opt.base_score))
portToC(agmp_opt, 'agmp_combined.c')

harus_opt.load_model("model-exports/aktuelle-exports/combined/harus_comb.json")
harus_opt.set_params(base_score = float(harus_opt.base_score),
                     num_parallel_tree = 1)
print(float(harus_opt.base_score))
portToC(harus_opt, 'harus_combined.c')

semu_opt.load_model("model-exports/aktuelle-exports/combined/semu_comb.json")
semu_opt.set_params(base_score = float(semu_opt.base_score))
print(float(semu_opt.base_score))
portToC(semu_opt, 'semu_combined.c')

0.49990478
Model exported to: "model-exports/plainC/agmp_combined.c"
0.5
Model exported to: "model-exports/plainC/harus_combined.c"
0.55469894
Model exported to: "model-exports/plainC/semu_combined.c"
